# Этап 3. Эксперименты с моделью

Данные после очистки лежат в `clean_flats_dataset`. Тут смотрю, что с ними можно сделать, и подбираю модель - потом этот же код переезжает в `scripts/`.

Перед запуском в папке `part2_dvc` должен быть файл `.env` с доступами к личной БД (шаблон - `.env_template`).

In [1]:
import os
from urllib.parse import quote_plus

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor

pd.set_option('display.max_columns', 50)

/var/folders/4l/70xpw14x7kd1jvxmsc32pw1m0000gn/T/ipykernel_34508/928473109.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


## 1. Читаю данные из личной БД

Очищенный датасет лежит в таблице `clean_flats_dataset`.

In [2]:
load_dotenv()

host = os.environ['DB_DESTINATION_HOST']
port = os.environ['DB_DESTINATION_PORT']
db_name = os.environ['DB_DESTINATION_NAME']
user = os.environ['DB_DESTINATION_USER']
# пароль прогоняю через quote_plus: спецсимволы в нём ломают строку подключения
password = quote_plus(os.environ['DB_DESTINATION_PASSWORD'])

engine = create_engine(f'postgresql://{user}:{password}@{host}:{port}/{db_name}')

data = pd.read_sql('select * from clean_flats_dataset', engine)
print('строк и колонок:', data.shape)
data.head()

строк и колонок: (82877, 19)


,id,flat_id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
0,1,0,6220,9,9.90,19.900000,1,False,False,35.099998,9500000.0,1965,6,55.717113,37.781120,2.64,84,12,True
1,2,2,17821,9,9.00,32.000000,2,False,False,56.000000,13500000.0,2000,4,55.740040,37.761742,2.70,80,10,True
2,3,3,18579,1,10.10,43.099998,3,False,False,76.000000,20000000.0,2002,4,55.672016,37.570877,2.64,771,17,True
3,4,4,9293,3,3.00,14.000000,1,False,False,24.000000,5200000.0,1971,1,55.808807,37.707306,2.60,208,9,True
4,5,6,5576,1,6.18,29.340000,2,False,False,44.520000,9500000.0,1964,4,55.795589,37.722622,2.64,180,5,False


## 2. Смотрю, что внутри

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82877 entries, 0 to 82876
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 82877 non-null  int64  
 1   flat_id            82877 non-null  int64  
 2   building_id        82877 non-null  int64  
 3   floor              82877 non-null  int64  
 4   kitchen_area       82877 non-null  float64
 5   living_area        82877 non-null  float64
 6   rooms              82877 non-null  int64  
 7   is_apartment       82877 non-null  bool   
 8   studio             82877 non-null  bool   
 9   total_area         82877 non-null  float64
 10  price              82877 non-null  float64
 11  build_year         82877 non-null  int64  
 12  building_type_int  82877 non-null  int64  
 13  latitude           82877 non-null  float64
 14  longitude          82877 non-null  float64
 15  ceiling_height     82877 non-null  float64
 16  flats_count        828

In [4]:
data.describe().T

,count,mean,std,min,25%,50%,75%,max
id,82877.0,4.143900e+04,2.392467e+04,1.000000,2.072000e+04,4.143900e+04,6.215800e+04,8.287700e+04
flat_id,82877.0,6.862641e+04,4.111185e+04,0.000000,3.290900e+04,6.737000e+04,1.037460e+05,1.413610e+05
building_id,82877.0,1.387263e+04,5.760212e+03,398.000000,9.166000e+03,1.351300e+04,1.851500e+04,2.462000e+04
floor,82877.0,7.179073e+00,4.888798e+00,1.000000,3.000000e+00,6.000000e+00,1.000000e+01,4.200000e+01
kitchen_area,82877.0,8.516164e+00,2.161487e+00,1.500000,6.500000e+00,8.500000e+00,1.000000e+01,1.584000e+01
living_area,82877.0,2.763463e+01,1.277581e+01,0.000000,1.900000e+01,2.820000e+01,3.440000e+01,6.900000e+01
rooms,82877.0,1.895725e+00,7.992924e-01,1.000000,1.000000e+00,2.000000e+00,2.000000e+00,5.000000e+00
total_area,82877.0,5.017901e+01,1.488698e+01,11.500000,3.810000e+01,4.600000e+01,5.900000e+01,1.200000e+02
price,82877.0,1.102065e+07,3.481494e+06,1000000.000000,8.400000e+06,1.050000e+07,1.310000e+07,2.100000e+07
build_year,82877.0,1.985935e+03,1.761842e+01,1926.000000,1.970000e+03,1.982000e+03,2.002000e+03,2.023000e+03


In [5]:
# после очистки пропусков быть не должно
print('пропусков всего:', data.isna().sum().sum())
print('строк:', len(data), ', уникальных flat_id:', data['flat_id'].nunique())

пропусков всего: 0
строк: 82877 , уникальных flat_id: 82877


In [6]:
data['price'].hist(bins=50, figsize=(8, 4))
plt.title('Распределение цены')
plt.xlabel('цена, руб.')
plt.show()

<Figure size 800x400 with 1 Axes>

Цена распределена неравномерно: недорогих квартир много, дорогих мало, у гистограммы длинный правый хвост. Это пригодится дальше, когда буду выбирать основную метрику.

## 3. Признаки и целевая переменная

Целевая переменная - `price`, цена квартиры в рублях. Значит, это задача регрессии.

`id`, `flat_id` и `building_id` выкидываю - это номера, цену они не объясняют, а всё полезное про дом лежит в отдельных колонках.

Категориальные признаки - `building_type_int` и три флага `is_apartment`, `studio`, `has_elevator`. Тип дома записан числом, но это код, а не величина: тип 5 не больше типа 1 в пять раз, поэтому его нужно кодировать, а не масштабировать. Перед кодированием перевожу категориальные колонки в строки, чтобы значения из БД (bool или int) и значения из csv-файлов пайплайна выглядели одинаково.

Все остальные колонки числовые. Эти же списки записаны в `params.yaml`.

In [7]:
target_col = 'price'
drop_cols = ['id', 'flat_id', 'building_id']
cat_cols = ['building_type_int', 'is_apartment', 'studio', 'has_elevator']

X = data.drop(columns=drop_cols + [target_col])
y = data[target_col]

X[cat_cols] = X[cat_cols].astype(str)
num_cols = [col for col in X.columns if col not in cat_cols]

print('категориальные признаки:', cat_cols)
print('числовые признаки:', num_cols)

категориальные признаки: ['building_type_int', 'is_apartment', 'studio', 'has_elevator']
числовые признаки: ['floor', 'kitchen_area', 'living_area', 'rooms', 'total_area', 'build_year', 'latitude', 'longitude', 'ceiling_height', 'flats_count', 'floors_total']


## 4. Делю данные на train и test

Откладываю 20% строк на тест: на них модель не учится, поэтому по ним честнее видно качество. `random_state` фиксирую, чтобы разбиение повторялось. Те же два числа лежат в `params.yaml` и используются в `scripts/split.py`.

In [8]:
test_size = 0.2
random_state = 42

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
print('train:', X_train.shape, 'test:', X_test.shape)

train: (66301, 15) test: (16576, 15)


## 5. Какие метрики считаю

Главной беру MAE - она в рублях, её видно без пересчёта. RMSE тут хуже: хвост по цене длинный, и несколько дорогих квартир перетягивают её на себя. RMSE всё равно печатаю, на ней учится CatBoost (`loss_function: RMSE`). MAPE и R2 добавил просто чтобы было с чем сравнить бейзлайн: у константы R2 почти ноль.

## 6. Бейзлайн: медианная цена

Самая простая модель - предсказывать всем квартирам одно и то же число, медиану цены по обучающей выборке. Если нормальная модель не обгонит бейзлайн, значит она ничего полезного не выучила. Медиану беру, а не среднее, потому что из-за дорогих квартир среднее завышено.

In [9]:
baseline = DummyRegressor(strategy='median')
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

baseline_metrics = {
    'mae': round(mean_absolute_error(y_test, baseline_pred), 2),
    'rmse': round(mean_squared_error(y_test, baseline_pred) ** 0.5, 2),
    'mape': round(mean_absolute_percentage_error(y_test, baseline_pred), 4),
    'r2': round(r2_score(y_test, baseline_pred), 4),
}
baseline_metrics

{'mae': 2772032.94, 'rmse': 3519012.36, 'mape': 0.2644, 'r2': -0.0243}

## 7. Базовая модель CatBoost

Собираю пайплайн из двух частей. Сначала `ColumnTransformer`: категориальные колонки кодирует `OneHotEncoder` (для флагов из двух значений хватает одного столбца, поэтому `drop='if_binary'`, а `handle_unknown='ignore'` - на случай категории, которой не было в обучении), числовые приводит к одному масштабу `StandardScaler`. Дальше идёт `CatBoostRegressor`. Точно такой же пайплайн собирается в `scripts/fit.py`, а его параметры вынесены в `params.yaml`.

In [10]:
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(drop='if_binary', handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', StandardScaler(), num_cols),
])

model = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                          loss_function='RMSE', verbose=0, thread_count=-1,
                          random_seed=random_state)

pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
pipeline.fit(X_train, y_train)

catboost_pred = pipeline.predict(X_test)
catboost_metrics = {
    'mae': round(mean_absolute_error(y_test, catboost_pred), 2),
    'rmse': round(mean_squared_error(y_test, catboost_pred) ** 0.5, 2),
    'mape': round(mean_absolute_percentage_error(y_test, catboost_pred), 4),
    'r2': round(r2_score(y_test, catboost_pred), 4),
}
catboost_metrics

{'mae': 1666729.09, 'rmse': 2063365.41, 'mape': 0.1619, 'r2': 0.6479}

In [11]:
# обе модели на одной тестовой выборке
pd.DataFrame({'бейзлайн (медиана)': baseline_metrics, 'catboost': catboost_metrics}).T

,mae,rmse,mape,r2
бейзлайн (медиана),2772032.94,3519012.36,0.2644,-0.0243
catboost,1666729.09,2063365.41,0.1619,0.6479


## 8. Важность признаков

Смотрю, на что модель опирается сильнее всего. Если бы в верхних строчках оказался идентификатор или колонка, которая по смыслу на цену не влияет, это был бы повод ещё раз проверить данные.

In [12]:
feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()
importances = pipeline.named_steps['model'].get_feature_importance()

pd.Series(importances, index=feature_names).sort_values(ascending=False).head(15)

num__total_area             33.783516
num__latitude               27.891822
num__longitude              24.568440
num__floors_total            2.327563
num__build_year              2.321552
num__kitchen_area            2.263932
num__rooms                   1.823256
num__ceiling_height          1.404422
num__floor                   1.239767
num__living_area             0.984619
num__flats_count             0.601833
cat__building_type_int_2     0.252902
cat__building_type_int_4     0.200320
cat__building_type_int_1     0.172043
cat__has_elevator_True       0.086212
dtype: float64

## Выводы

MAE у медианы 2.77 млн, у CatBoost 1.67 млн - почти в 1.7 раза меньше. R2 у бейзлайна вообще ушёл в минус (-0.02), у модели 0.65, MAPE 26% против 16%. То есть признаки цену объясняют, обучение не впустую.

Основной метрикой оставляю MAE: она в рублях и не перекашивается парой очень дорогих квартир.

Три признака забирают почти всю важность: площадь (34), широта и долгота (28 и 25). Дальше сразу обрыв - у этажности и года постройки уже по 2 с небольшим. Выходит, модель смотрит в основном на размер квартиры и на то, где стоит дом. Ничего странного наверху нет, идентификаторы я выкинул заранее.

Дальше этот же код разложен по `scripts/`, порядок шагов и зависимости - в `dvc.yaml`, параметры - в `params.yaml`. `iterations` и `learning_rate` подбирал на глаз, надо будет вернуться и попробовать перебор.